In [1]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
import warnings
import torch
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = "/content/drive/MyDrive/Colab Notebooks/efsa_sentiment_classification.csv"
df = pd.read_csv(file_path)
# Extract test
df = df[939:].reset_index(drop=True)

In [18]:
!pip install -U bitsandbytes

In [19]:
coarse_event_definitions = """
- Financial Affairs: Earnings announcements, forecasts, or other financial updates.
- Shareholder Affairs: Actions by shareholders such as pledges, stock adjustments.
- Stock Affairs: Stock buybacks, dividends, price movement announcements.
- Business Operations: New products, partnerships, quality control, or business activities.
- Compliance and Credit: Legal issues, investigations, rating changes.
- Management Affairs: Executive hires, promotions, or organizational changes.
- Financing and Investment: Mergers, acquisitions, funding rounds, IPOs.
"""

In [4]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
# MODEL_NAME = "mistralai/Mixtral-8x7B-Instruct-v0.1"

SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

FINE_TO_COARSE = {}
FINE_EVENT_LIST = []

for coarse_key, fine_list in FINANCIAL_EVENT_LIST.items():
      for fine_event in fine_list:
          FINE_TO_COARSE[fine_event] = coarse_key
          FINE_EVENT_LIST.append(fine_event)

In [6]:
INDUSTRY_LIST = ["Energy Equipment & Services", "Oil, Gas & Consumable Fuels", "Chemicals", "Construction Materials", "Containers & Packaging", "Metals & Mining", "Paper & Forest Products", "Aerospace & Defense", "Building Products", "Construction & Engineering", "Electrical Equipment", "Industrial Conglomerates", "Machinery", "Trading Companies & Distributors", "Commercial Services & Supplies", "Professional Services", "Air Freight & Logistics", "Passenger Airlines", "Marine Transportation", "Ground Transportation", "Transportation Infrastructure", "Automobile Components", "Automobiles", "Household Durables", "Leisure Products", "Textiles, Apparel & Luxury Goods", "Hotels, Restaurants & Leisure", "Diversified Consumer Services", "Distributors", "Broadline Retail", "Specialty Retail", "Consumer Staples Distribution & Retail", "Beverages", "Food Products", "Tobacco", "Household Products", "Personal Care Products", "Health Care Equipment & Supplies", "Health Care Providers & Services", "Health Care Technology", "Biotechnology", "Pharmaceuticals", "Life Sciences Tools & Services", "Banks", "Financial Services", "Consumer Finance", "Capital Markets", "Mortgage Real Estate Investment Trusts (REITs)", "Insurance", "IT Services", "Software", "Communications Equipment", "Technology Hardware, Storage & Peripherals", "Electronic Equipment, Instruments & Components", "Semiconductors & Semiconductor Equipment", "Diversified Telecommunication Services", "Wireless Telecommunication Services", "Media", "Entertainment", "Interactive Media & Services", "Electric Utilities", "Gas Utilities", "Multi-Utilities", "Water Utilities", "Independent Power and Renewable Electricity Producers", "Diversified REITs", "Industrial REITs", "Hotel & Resort REITs", "Office REITs", "Health Care REITs", "Residential REITs", "Retail REITs", "Specialized REITs", "Real Estate Management & Development"]

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Loading model and tokenizer...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype="auto", quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer", use_fast=True)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/Colab Notebooks/mistral_lora_model")
print("Model loaded successfully!")

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!


In [8]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [12]:
logger = logging.getLogger(__name__)

def query_model(model, model_name, tokenizer, prompt: str, max_new_tokens=512, temperature=0.7) -> str:
    """Generate model response for given prompt using Mistral Instruct format."""

    if 'mistral' in model_name:
        instruction = f"[INST] {prompt} [/INST]"
    elif 'llama' in model_name:
        instruction = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n"
            f"{prompt}\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
        )

    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
    try:
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                    temperature=temperature, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Remove prompt from response to get just the answer:
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""

def extract_companies(text: str) -> str:
    """Extract company names from financial text."""
    prompt = (f"Financial text: {text}\n\n"
              "Extract all full company names mentioned in the text. "
              "Only include actual companies, not individuals or other institutions. "
              "Return a comma-separated list of company names only, no extra text, new lines, or punctuation. "
        )

    try:
        response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.2)

        # Catch cases when response is not clean
        cleaned = response.replace('\n', ',').replace(';', ',')
        companies = [company.strip() for company in cleaned.split(',') if company.strip()]
        return companies if companies else ["Unknown Company"]
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return "Unknown Company"


def get_industry(text: str, company_name: str) -> str:
    """Get industry for a company using model prompt."""
    prompt = (f"Text: {text}\n"
            f"Company: {company_name}\n\n"
            f"As if consulting Bloomberg Terminal, FactSet, or Yahoo Finance, determine {company_name}'s industry.\n\n"
            f"Which GICS sector best fits?\n\n"
            f"Available GICS sectors: {INDUSTRY_LIST}\n\n"
            "Respond only with one industry name from the list, without additional text or punctuation."
        )

    try:
        response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.4)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"


def classify_coarse_grained_event(text: str, industry: str, company_name: str) -> Tuple[str, Any]:
    """Classify the coarse-grained event type."""
    coarse_events = list(FINANCIAL_EVENT_LIST.keys())
    prompt = (f"Text: \"{text}\"\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"Choose the most appropriate coarse-grained event type from this list based on what is explicitly stated in the text: {coarse_events}\n\n"
              # f"The following are the event definitions: {coarse_event_definitions}"
              "Please only answer with one coarse-grained event type, without additional text or punctuation."
          )

    try:
        response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.1)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Event"
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event"


def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str) -> Tuple[str, Any]:
    """Classify the fine-grained event type."""
    if coarse_event in FINANCIAL_EVENT_LIST:
        fine_events = FINANCIAL_EVENT_LIST[coarse_event]
        prompt = (f"The following news text was classified as the event category: {coarse_event}.\n\n"
                  f"Text: \"{text}\"\n"
                  f"Company: {company_name}\n"
                  f"Industry: {industry}\n\n"
                  f"Now select the specific fine-grained event that best describes what happened, based on this list {fine_events}.\n\n"
                  "If the event doesn't fit perfectly into a specific event type, choose the most relevant 'Other' category.\n"
                  "Please only answer with one fine-grained event type, without additional text or punctuation."
            )
    else:
        return "Unknown Fine Event"

    try:
        response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.3)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Fine Event"
    except Exception as e:
        return "Unknown Fine Event"

def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str) -> str:
    """Analyze sentiment polarity of the event."""
    prompt = (f"You are a financial sentiment model. Analyze the sentiment expressed in the text.\n\n"
            f"Company: {company_name} ({industry})\n"
            f"Event: {fine_event}\n"
            f"News: {text}\n\n"
            # f"- POS = The sentence includes positive or optimistic language.\n"
            # f"- NEG = The sentence includes negative or pessimistic language.\n"
            "Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."
        )

    try:
        response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.5)
        return response.strip() or "Unknown"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "Unknown"

In [10]:
def analyze_financial_text(text: str) -> Tuple[str, str, str, str, str]:
    """
    Analyze a single financial text and return the quintuple.

    Returns:
        Tuple[str, str, str, str, str]: (company_name, industry, coarse_event, fine_event, sentiment)
    """
    try:
        # Extract all company names
        company_names = extract_companies(text)
        results = []

        for company_name in company_names:
            try:
                # Get industry
                industry = get_industry(text, company_name)

                # Classify fine-grained event first, then derive coarse event
                coarse_event = classify_coarse_grained_event(
                    text, company_name, industry)

                fine_event = classify_fine_grained_event(
                    text, industry, company_name, coarse_event)

                # Analyze sentiment
                sentiment = analyze_sentiment(
                    text, industry, company_name, coarse_event, fine_event)

                results.append((text, company_name, industry,
                               coarse_event, fine_event, sentiment))

            except Exception as e:
                logger.error(
                    f"Error analyzing text for company {company_name}: {e}")
                results.append(
                    (company_name, "Error", "Error", "Error", "Error"))

        return results

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return [("Error", "Error", "Error", "Error", "Error")]

In [13]:
results = []

# Process each text
for idx, row in df.iterrows():
    text = row['Sentence']

    if pd.isna(text) or text.strip() == '':
        logger.warning(f"Empty text at index {idx}")
        results.append(("", "", "", "", "", ""))
        continue

    print(f"Processing {idx+1}/{len(df)}: {text[:50]}...")

    # Analyze the text
    result = analyze_financial_text(text)

    # Store results
    results.append(result)

    # Print progress every 10 items
    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1}/{len(df)} texts")

Processing 1/234: $TSLA TO RECALL 2,700 MODEL X SUVS FOR SEAT-BELT F...
Processing 2/234: 5 Best Analyst Rated Stocks in Last 72hrs: $ORCL $...
Processing 3/234: Bought some more $CELG as it is ready for a bounce...
Processing 4/234: Analysts impressed with progress at Tesla's flagsh...
Processing 5/234: $CELG back near 104, where it opened and rallied h...
Processing 6/234: $AMZN new HOD with conviction keeping $570 on watc...
Processing 7/234: $FB  rejecting HIGHS shortable...at 109...
Processing 8/234: $CSX is up today to report.  Wall Street is expect...
Processing 9/234: $FB turns back down in early trading.... https://t...
Processing 10/234: Followed the levels I shared with you on $NFLX $GO...
Completed 10/234 texts
Processing 11/234: $FAST $GWW - daily sales slowing again, pretty tim...
Processing 12/234: Tesla Motors recalls 2,700 Model X SUVs $TSLA http...


KeyboardInterrupt: 

In [14]:
# Flatten the list of lists and add the original sentence to each entry
flat_results_with_sentence = []
for i, sentence_results in enumerate(results):
    for company_result in sentence_results:
        flat_results_with_sentence.append(company_result)

results_df = pd.DataFrame(flat_results_with_sentence, columns=['Sentence', 'Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'])
results_df['Sentiment'] = results_df['Sentiment'].str[:3]

In [15]:
results_df.head(20)

,Sentence,Company,Industry,Coarse-Grained Event,Fine-Grained Event,Sentiment
0,"$TSLA TO RECALL 2,700 MODEL X SUVS FOR SEAT-BE...",Tesla,Ground Transportation,Business Operations,"Sales, Market Share Changes",NEG
1,5 Best Analyst Rated Stocks in Last 72hrs: $OR...,ORCL,Real Estate Management & Development,Business Operations,"Sales, Market Share Changes",POS
2,5 Best Analyst Rated Stocks in Last 72hrs: $OR...,AAPL,Financial Services,Business Operations,"Sales, Market Share Changes",NEU
3,5 Best Analyst Rated Stocks in Last 72hrs: $OR...,CBS,Media,Business Operations,"Sales, Market Share Changes",NEU
4,5 Best Analyst Rated Stocks in Last 72hrs: $OR...,INO,Diversified REITs,Business Operations,"Sales, Market Share Changes",NEU
5,5 Best Analyst Rated Stocks in Last 72hrs: $OR...,CPXX,Diversified REITs,Business Operations,"Sales, Market Share Changes",POS
6,Bought some more $CELG as it is ready for a bo...,CELG,Pharmaceuticals,Business Operations,"Sales, Market Share Changes",NEU
7,Analysts impressed with progress at Tesla's fl...,$TSLA,Diversified REITs,Business Operations,"Sales, Market Share Changes",NEU
8,"$CELG back near 104, where it opened and ralli...",Celgene,Pharmaceuticals,Business Operations,"Sales, Market Share Changes",POS
9,$AMZN new HOD with conviction keeping $570 on ...,AMZN,Real Estate Management & Development,Business Operations,"Sales, Market Share Changes",NEU


In [24]:
from collections import defaultdict, Counter
from sklearn.metrics import f1_score, accuracy_score

def calculate_eval_metrics_by_label(df, results_df):
    """
    Calculate macro F1, weighted F1, and accuracy for each category.
    """
    preds_by_sentence = results_df.groupby('Sentence')
    categories = ['Sentiment', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event']

    gt_labels = defaultdict(list)
    pred_labels = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for cat in categories:
                gt_labels[cat].append(str(row[cat]))
                pred_labels[cat].append('NO_PREDICTION')
            continue

        preds = preds_by_sentence.get_group(sentence)
        company_match = preds[preds['Company'].str.lower().str.contains(gt_company, na=False)]

        if company_match.empty:
            for cat in categories:
                gt_labels[cat].append(str(row[cat]))
                pred_labels[cat].append('NO_COMPANY_MATCH')
            continue

        pred_row = company_match.iloc[0]

        for cat in categories:
            gt_val = str(row[cat])
            pred_val = str(pred_row[cat])
            gt_labels[cat].append(gt_val)
            pred_labels[cat].append(pred_val)

    # Compute all metrics
    results = {}
    for cat in categories:
        y_true = gt_labels[cat]
        y_pred = pred_labels[cat]
        labels = list(set(y_true + y_pred))

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)

        # Calculate F1 scores
        macro_f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0)

        results[cat] = {
            'accuracy': round(accuracy, 4),
            'f1_macro': round(macro_f1, 4),
            'f1_weighted': round(weighted_f1, 4)
        }

    return results

In [25]:
calculate_eval_metrics_by_label(df, results_df)

{'Sentiment': {'accuracy': 0.0171, 'f1_macro': 0.0126, 'f1_weighted': 0.0336},
 'Industry': {'accuracy': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0},
 'Coarse-Grained Event': {'accuracy': 0.0043,
  'f1_macro': 0.0016,
  'f1_weighted': 0.0085},
 'Fine-Grained Event': {'accuracy': 0.0043,
  'f1_macro': 0.0011,
  'f1_weighted': 0.0085}}

In [26]:
from sklearn.metrics import f1_score
from collections import defaultdict

def calculate_eval_metrics_by_variant(df, results_df):
    """
    Calculate macro F1, weighted F1, and accuracy for EFSA variants.
    """
    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine-Grained Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment']
    }

    preds_by_sentence = results_df.groupby('Sentence')

    lab_true = defaultdict(list)
    lab_pred = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_PREDICTION'] * len(cols)))
            continue

        preds = preds_by_sentence.get_group(sentence)

        company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_COMPANY_MATCH'] * len(cols)))
            continue

        pred_row = company_match.iloc[0]

        for variant, cols in variants.items():
            gt_string = "|".join(str(row[col]) for col in cols)
            pred_string = "|".join(str(pred_row[col]) for col in cols)
            lab_true[variant].append(gt_string)
            lab_pred[variant].append(pred_string)

    results = {}
    for variant in variants:
        true = lab_true[variant]
        pred = lab_pred[variant]
        labels = list(set(true + pred))

        # Calculate accuracy
        accuracy = accuracy_score(true, pred)

        # Calculate F1 scores
        macro_f1 = f1_score(true, pred, average='macro', labels=labels, zero_division=0)
        weighted_f1 = f1_score(true, pred, average='weighted', labels=labels, zero_division=0)

        results[variant] = {
            'accuracy': round(accuracy, 4),
            'f1_macro': round(macro_f1, 4),
            'f1_weighted': round(weighted_f1, 4)
        }

    return results

# Helper function to print results in a nice format
def print_comprehensive_results(by_label_results, by_variant_results):
    """
    Print the results in a formatted way for easy reading.
    """
    print("=" * 60)
    print("RESULTS BY LABEL")
    print("=" * 60)

    for category, metrics in by_label_results.items():
        print(f"\n{category}:")
        print(f"  Accuracy:     {metrics['accuracy']}")
        print(f"  Macro F1:     {metrics['f1_macro']}")
        print(f"  Weighted F1:  {metrics['f1_weighted']}")

    print("\n" + "=" * 60)
    print("RESULTS BY VARIANT")
    print("=" * 60)

    for variant, metrics in by_variant_results.items():
        print(f"\n{variant}:")
        print(f"  Accuracy:     {metrics['accuracy']}")
        print(f"  Macro F1:     {metrics['f1_macro']}")
        print(f"  Weighted F1:  {metrics['f1_weighted']}")


In [28]:
calculate_eval_metrics_by_variant(df, results_df)

{'EFSA': {'accuracy': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0},
 'Coarse-grained EFSA': {'accuracy': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0},
 'Fine-grained EFSA': {'accuracy': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0},
 'Entity-Level FSA': {'accuracy': 0.0, 'f1_macro': 0.0, 'f1_weighted': 0.0}}

In [ ]:
def calculate_efsa_variant_accuracy(df, results_df):
    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine-Grained Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment']
    }

    preds_by_sentence = results_df.groupby('Sentence')

    lab_true = defaultdict(list)
    lab_pred = defaultdict(list)

    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()

        if sentence not in preds_by_sentence.groups:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_PREDICTION'] * len(cols)))
            continue

        preds = preds_by_sentence.get_group(sentence)

        company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]

        if company_match.empty:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_COMPANY_MATCH'] * len(cols)))
            continue

        pred_row = company_match.iloc[0]

        for variant, cols in variants.items():
            gt_string = "|".join(str(row[col]) for col in cols)
            pred_string = "|".join(str(pred_row[col]) for col in cols)
            lab_true[variant].append(gt_string)
            lab_pred[variant].append(pred_string)

    results = {}
    for variant in variants:
        true = lab_true[variant]
        pred = lab_pred[variant]
        accuracy = accuracy_score(true, pred)
        results[variant] = round(accuracy, 4)

    return results

In [ ]:
calculate_efsa_variant_accuracy(df, results_df)

In [ ]:
def classify_fine_grained_event_first(text: str, industry: str, company_name: str) -> Tuple[str, str, Any]:
    """Classify fine-grained event first, then map to coarse."""

    prompt = (f"Financial text: {text}\n"
            f"Company: {company_name}\n"
            f"Industry: {industry}\n\n"
            f"As a financial sentiment model, determine the specific type of corporate event that occurred at {company_name}. "
            f"Select the most precise event type from this comprehensive list: {FINE_EVENT_LIST}\n\n"
            "If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.\n"
            "Respond only with an event type from the provided list, without additional text, quotation marks, or punctuation."
        )

    try:
        fine_response = query_model(model, MODEL_NAME, tokenizer, prompt, temperature=0.5).strip()

        # Clean quotation marks
        fine_response = fine_response.replace('"', '').replace("'", "")

        # Map to coarse event
        coarse_event = FINE_TO_COARSE.get(fine_response, "Unknown Event")

        return coarse_event, fine_response
    except Exception as e:
        logger.error(f"Error classifying fine-grained event first: {e}")
        return "Unknown Event", "Unknown Fine Event"